# Preprocessing x sensor placement -> latent space

What the AER sees is `client_datasets` -> `preprocess_clients` -> `(N, W, F)` windows. Two things
upstream of that were unexamined: **where** the channels come from (placement, never wired into
the FL path) and **what** is done to them in time.

| block | question | trains? |
|---|---|---|
| SS.2-4 | what am I feeding the model | no |
| SS.5 | which signal component does each transform keep (level / amplitude / shape) | no |
| SS.6-10 | grid `placement x preprocessing`, scored on latent + drift | yes |
| SS.11 | does the ranking replicate across worlds | yes |
| SS.12 | latent space, per placement, per round | no |
| SS.13-14 | client similarity and clustering, **init vs final** phase | no |

`preprocess_clients` and `FPLTrainer` are imported, never re-implemented; placement enters as a
column selection, no re-simulation; ground truth enters at scoring time only.

**Metric decisions this version encodes** (from the runs so far):

- Similarity structure is read as **`regime_gap`** (within- minus between-regime mean similarity),
  continuous, against its own **label-shuffle null**. ARI is kept as a secondary column only: with
  5 clients and 2 regime tokens it takes ~8 discrete values, so it mangles real movement.
- **`mean_offdiag` is dropped.** After `_deconfounded_prototypes` the client vectors are
  near-centered, which pins the mean off-diagonal cosine at about `-1/(N-1)`. It measured a
  constant. Degeneracy is now flagged by *low spread* (`sd_offdiag`).
- **Only the `dom` (prototype) channel is scored.** `delta_first` is ~0 in the first months, so
  `S_drift` and any fusion containing it are noise in the `init` phase.
- **Migration is one sanity column, not a section** (SS.13): the cluster-crossing test had no
  power -- most clients "moved" every round because the clustering was re-cut on a near-identical
  matrix (`rank_stability` up to 1.00). What survives is denominator-free: does the mover's
  similarity shift rank first among clients.
- Every p-value here has a **floor of 1/n_assignments** (0.10 for 5 clients / 2 tokens). Rank by
  effect size; significance needs a 3-token world.

## 0. Setup

In [1]:
from __future__ import annotations
import copy, itertools, json, pathlib, sys, time, warnings

import numpy as np, pandas as pd, torch, yaml
warnings.filterwarnings("ignore")

HERE = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "conf" / "base" / "parameters.yml").exists()
             and (p / "src" / "fedwater").is_dir()), None)
assert ROOT is not None, "run this from inside the fedwater repo (notebooks/...)"
for p in (ROOT / "src", ROOT / "notebooks", ROOT, HERE):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from notebooks.SensorPlacement import placement_poc as P     
from fedwater.pipelines.fl_preprocessing.nodes import preprocess_clients
from fedwater.pipelines.fl_training.federated import FPLTrainer
from fedwater.pipelines.fl_training.nodes import effective_fl_seed
from fedwater.pipelines.drift_detection.nodes import compute_drift_signals
from fedwater.pipelines.dependence_oracle.nodes import deseasonalize

BASE_PARAMS = yaml.safe_load((ROOT / "conf" / "base" / "parameters.yml").read_text())
SEED   = BASE_PARAMS.get("seed", 42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SANDBOX = HERE / "latent_sandbox"; SANDBOX.mkdir(exist_ok=True)
pd.set_option("display.width", 170)

print(f"root {ROOT}\ndevice {DEVICE}  seed {SEED}  sandbox {SANDBOX}")

root C:\Users\arthu\USPy\10_Mestrado\fedWater
device cuda  seed 42  sandbox C:\Users\arthu\USPy\10_Mestrado\fedWater\notebooks\prospection\latent_sandbox


## 1. World and configuration

`discover_worlds` reads the experiments cache directly. Only `baseline` worlds are used: under
`partial`/`isolated` a topology-driven placement sees the same pipes the labels come from.

In [2]:
index = P.discover_worlds(ROOT)
assert len(index) and index["usable"].any(), "no usable cached worlds"
usable = index[index["usable"] & (index["variant"] == "baseline")]
display(usable[["tag", "sim_seed", "consumption_map", "n_months", "drift_district"]]
        .reset_index(drop=True))


# --- manifests are not guaranteed complete: any of resolution_h / n_months /
# --- days_per_month can be None, and `int(None)` then kills every cell in the world.
# --- Everything downstream (including preprocess_clients, which receives TIME) reads
# --- the coerced dict, so the repair happens once, here.
def _num(x, default=None):
    v = pd.to_numeric(x, errors="coerce")
    return default if pd.isna(v) else float(v)


def world_time(art: dict) -> dict:
    """`params.time` with every field numeric, recovered from the artifacts if absent."""
    t = dict(art["params"].get("time", {}) or {})
    res = _num(t.get("resolution_h"))
    if res is None or res <= 0:
        res = 24.0 / max(1.0, _num(art.get("steps_day"), 24.0))
    dpm = _num(t.get("days_per_month"), 30.0) or 30.0
    n = _num(t.get("n_months"))
    if n is None or n <= 0:                       # count it off the simulated horizon
        spm = max(1.0, (24.0 / res) * dpm)
        n = max(1.0, round(len(art["pressures"]) / spm))
    t.update(resolution_h=(int(res) if float(res).is_integer() else res),
             days_per_month=int(dpm), n_months=int(n))
    return t


def drift_start(world_dir, n_months=None) -> int:
    """First drifted month from gt_drift_schedule; midpoint if absent; 0 if unknowable."""
    f = pathlib.Path(world_dir) / "clone" / "data" / "03_primary" / "gt_drift_schedule.csv"
    if f.exists():
        try:
            d = pd.read_csv(f)
            col = next((c for c in d.columns if "month" in c.lower()), None)
            if col is not None and len(d):
                v = pd.to_numeric(d[col], errors="coerce").dropna()
                if len(v):
                    return int(v.min())
        except Exception:
            pass
    n = _num(n_months)
    return int(n) // 2 if n and n > 0 else 0


def world_fl(art: dict) -> dict:
    """`params.fl` with the fields we index numeric, and the device pinned."""
    fl = copy.deepcopy(art["params"].get("fl", BASE_PARAMS["fl"]))
    tr, pp = fl.setdefault("training", {}), fl.setdefault("preprocessing", {})
    tr["device"] = DEVICE
    for k, dflt in (("rounds", 15), ("local_epochs", 1), ("batch_size", 64)):
        tr[k] = int(_num(tr.get(k), dflt))
    for k, dflt in (("interval_agg_h", 2), ("window_size", 24), ("step_size", 6),
                    ("reference_months", 2)):
        pp[k] = int(_num(pp.get(k), dflt))
    return fl


WORLD = usable.iloc[-1]                       # <- the single-world study
ART   = P.load_world(WORLD["dir"])
TIME  = world_time(ART)
STEPS_DAY    = int(_num(ART.get("steps_day"), 24.0 / TIME["resolution_h"]))
FL_BASE      = world_fl(ART)
TRUTH_CLIENT = WORLD.get("drift_district")
DRIFT_START  = drift_start(WORLD["dir"], TIME["n_months"])
CURRENT_WORLD = WORLD["sim_hash"]            # which world CLIENTS/ART currently hold

print(f"{WORLD['tag']}  |  drift {TRUTH_CLIENT} from month {DRIFT_START}  |  "
      f"{TIME['n_months']} months @ {TIME['resolution_h']}h  |  steps/day {STEPS_DAY}")
raw_time = ART["params"].get("time", {}) or {}
if any(raw_time.get(k) is None for k in ("resolution_h", "days_per_month", "n_months")):
    print("  NOTE manifest had None in params.time "
          f"({ {k: raw_time.get(k) for k in ('resolution_h', 'days_per_month', 'n_months')} })"
          " -- values above were recovered from the artifacts")

,tag,sim_seed,consumption_map,n_months,drift_district
0,Clr__LR_LI_LI_LR_LI__baseline,42,LR_LI_LI_LR_LI,30,District_C
1,Dli__LI_LR_LI_LR_LI__baseline,42,LI_LR_LI_LR_LI,30,District_D
2,Bli__LI_LR_LI_LR_LI__baseline,42,LI_LR_LI_LR_LI,30,District_B
3,Ali__LR_LI_LI_LR_LI__baseline,42,LR_LI_LI_LR_LI,30,District_A
4,Blr__LR_LI_LI_LR_LI__baseline,42,LR_LI_LI_LR_LI,30,District_B
5,Elr__LI_LR_LI_LR_LI__baseline,42,LI_LR_LI_LR_LI,30,District_E
6,Blr__LI_LI_LR_LR_LR__baseline,42,LI_LI_LR_LR_LR,30,District_B
7,Alr__LI_LI_LR_LR_LR__baseline,42,LI_LI_LR_LR_LR,30,District_A
8,Dli__LR_LI_LI_LR_LI__baseline,42,LR_LI_LI_LR_LI,30,District_D
9,Clr__LI_LR_LI_LR_LI__baseline,42,LI_LR_LI_LR_LI,30,District_C


Dli__LI_LR_LR_LR_LI__baseline  |  drift District_D from month 6  |  30 months @ 1h  |  steps/day 24


## 2. Placement -> client datasets

`strategies()` returns a placement per rule; `build_series()` turns it into the sensor long frame.
The only new code is the repackaging into the wide `District_*.csv` schema
(`timestamp, month, p_*, q_*`) that `preprocess_clients` consumes -- this is the join that was
missing between the placement POC and the FL stack.

In [6]:
STRATEGIES = ["manual", "variance", "boundary"]      # per the placement POC finding

cands       = P.candidate_table(ART["wn"], ART["districts"])
feats, dist = P.topology_features(ART["wn"], ART["districts"], cands)
stats       = P.residual_stats(ART["pressures"], ART["flows"], cands, STEPS_DAY)
PLACEMENTS  = {k: v for k, v in
               P.strategies(cands, feats, stats, dist, ART["params"], seed=SEED).items()
               if k in STRATEGIES}


def clients_from_placement(art, placement, noise=False, seed=SEED,
                           kinds=None) -> dict[str, pd.DataFrame]:
    """placement -> {district: wide frame} in the client_datasets schema.

    kinds: subset of {"pressure","flow"} to keep (filters AFTER selection -- the candidates
    still competed under the full 2p+3q template). None keeps everything.
    """
    ss = P.build_series(art["pressures"], art["flows"], placement,
                        art["params"]["noise"] if noise else None, seed)
    if kinds:
        ss = ss[ss["kind"].isin(kinds)]
    wide  = ss.pivot(index="step", columns="sensor", values="observed").sort_index()
    month = ss.drop_duplicates("step").set_index("step")["month"].sort_index()
    key   = ss.drop_duplicates("sensor").set_index("sensor")["district"]
    res_h = world_time(art)["resolution_h"]          # coerced: manifests carry None
    ts    = pd.date_range("2020-01-01", periods=len(wide),
                          freq=f"{max(1, int(round(float(res_h))))}h")

    out = {}
    for d, grp in key.groupby(key):
        cols = sorted(grp.index)                       # p_* then q_*, stable across clients
        f = wide[cols].copy().reset_index(drop=True)
        f.insert(0, "month", month.to_numpy())
        f.insert(0, "timestamp", ts)
        out[d] = f
    return out


def assert_equal_channels(clients: dict[str, pd.DataFrame]):
    """FedAvg needs identical architectures -- every client must carry the same channel count."""
    n = {c: len([x for x in f.columns if x not in ("timestamp", "month")])
         for c, f in clients.items()}
    if len(set(n.values())) > 1:
        raise AssertionError(f"channel count differs across clients (breaks FedAvg): {n}")


CLIENTS = {s: clients_from_placement(ART, pl, kinds=("flow",)) for s, pl in PLACEMENTS.items()}
# CLIENTS = {s: clients_from_placement(ART, pl, kinds=("flow",))          # flow-only, 3 channels
#            for s, pl in PLACEMENTS.items()}
for s, frames in CLIENTS.items():
    assert_equal_channels(frames)

CHANNELS = {s: {c: [x for x in f.columns if x not in ("timestamp", "month")]
                for c, f in d.items()} for s, d in CLIENTS.items()}
pd.DataFrame({s: {c: ",".join(v) for c, v in ch.items()} for s, ch in CHANNELS.items()})

,manual,variance,boundary
District_A,"q_100,q_102,q_85","q_102,q_103,q_136","q_103,q_104,q_176"
District_B,"q_135,q_157,q_165","q_129,q_131,q_134","q_133,q_134,q_136"
District_C,"q_110,q_119,q_91","q_104,q_119,q_97","q_129,q_95,q_97"
District_D,"q_38,q_69,q_96","q_45,q_94,q_96","q_44,q_69,q_96"
District_E,"q_11,q_19,q_59","q_145,q_53,q_67","q_181,q_65,q_67"


## 3. The preprocessing stack

Two insertion points, deliberately kept apart:

* **before windowing** -- a transform on the client frame (`none | slog | diff | deseason_own |
  deseason_shared`). `preprocess_clients` then runs verbatim on the transformed frame.
* **after windowing** -- the scaling axis. `preprocess_clients` always fits a per-client MinMax
  on the commissioning months; `rescale_windows` inverts that affine map with `fl_scalers` and
  re-applies whatever we asked for, so `none` (physical units) and `zscore` are reachable
  without forking pipeline code.

In [7]:
def _cell_index(n, steps_day, month):
    day = np.arange(n) // steps_day
    return pd.DataFrame({"month": np.asarray(month), "hour": np.arange(n) % steps_day,
                         "weekend": (day % 7) >= 5})


def _slog(x):                       # flows are signed -- log1p on the magnitude
    return np.sign(x) * np.log1p(np.abs(x))


def deseason_shared(frames: dict, steps_day: int, k: int = 4) -> dict:
    """Remove the top-k PCs of the POOLED (month,hour,weekend) profile matrix.

    Each component is mean-zero across cells by construction, so a channel's own LEVEL
    survives -- the difference from `deseason_own`, which subtracts each channel's own cell
    means and therefore deletes exactly the level ladder the regime codes assert.
    """
    cols = {c: [x for x in f.columns if x not in ("timestamp", "month")] for c, f in frames.items()}
    any_f = next(iter(frames.values()))
    cells = _cell_index(len(any_f), steps_day, any_f["month"].to_numpy())
    gid = cells.groupby(["month", "hour", "weekend"], sort=True).ngroup().to_numpy()
    ncell = gid.max() + 1

    stack, names = [], []
    for c, f in frames.items():
        for x in cols[c]:
            v = f[x].to_numpy(float)
            stack.append(np.bincount(gid, v, ncell) / np.bincount(gid, minlength=ncell))
            names.append((c, x))
    Pm = np.column_stack(stack)                              # cells x channels
    mu, sd = Pm.mean(0), Pm.std(0, ddof=0)
    Z = (Pm - mu) / np.where(sd > 0, sd, 1.0)
    U, _, _ = np.linalg.svd(Z, full_matrices=False)
    B = U[:, :k]
    fitted = (B @ (B.T @ Z)) * np.where(sd > 0, sd, 1.0)     # channel units, mean-zero over cells

    out = {c: f.copy() for c, f in frames.items()}
    for j, (c, x) in enumerate(names):
        out[c][x] = out[c][x].to_numpy(float) - fitted[gid, j]
    return out


def apply_transform(frames: dict, name: str, steps_day: int) -> dict:
    if name == "none":
        return frames
    if name == "deseason_shared":
        return deseason_shared(frames, steps_day)
    out = {}
    for c, f in frames.items():
        g = f.copy()
        cols = [x for x in g.columns if x not in ("timestamp", "month")]
        if name == "slog":
            g[cols] = _slog(g[cols].to_numpy(float))
        elif name == "diff":
            g[cols] = g[cols].diff().fillna(0.0)
        elif name == "deseason_own":            # fedwater invariant 5, month-aware
            wide = g[cols].copy()
            wide.index = np.arange(len(g))
            res = deseasonalize(wide, steps_day, g["month"].copy(), month_aware=True)
            g[cols] = np.asarray(res)
        else:
            raise KeyError(name)
        out[c] = g
    return out


TRANSFORMS = ["none", "slog", "diff", "deseason_own", "deseason_shared"]

In [8]:
def _scaler_bounds(scalers: pd.DataFrame, client: str, sensors) -> tuple[np.ndarray, np.ndarray]:
    """(min, max) per channel from fl_scalers, whatever it calls its columns."""
    low = {c.lower(): c for c in scalers.columns}
    ccol = next(low[k] for k in ("district", "client", "partition", "name") if k in low)
    scol = next(low[k] for k in ("sensor", "feature", "channel", "column") if k in low)
    mncol = next(c for c in scalers.columns if "min" in c.lower())
    mxcol = next(c for c in scalers.columns if "max" in c.lower())
    g = scalers[scalers[ccol] == client].drop_duplicates(scol).set_index(scol)
    return (g.loc[list(sensors), mncol].to_numpy(float),
            g.loc[list(sensors), mxcol].to_numpy(float))


def rescale_windows(fl_windows: dict, scalers: pd.DataFrame, mode: str, fl: dict) -> dict:
    """minmax_ref (as-is) | none (physical units) | zscore (per client-channel)."""
    if mode == "minmax_ref":
        return fl_windows
    lo, hi = fl["preprocessing"]["feature_range"]
    out = {}
    for c, d in fl_windows.items():
        mn, mx = _scaler_bounds(scalers, c, d["sensors"])
        raw = (d["windows"].astype(np.float64) - lo) / (hi - lo) * (mx - mn) + mn
        if mode == "none":
            w = raw
        elif mode == "zscore":
            ref = d["labels"] < fl["preprocessing"]["reference_months"]
            src = raw[ref] if ref.any() else raw
            mu = src.reshape(-1, src.shape[-1]).mean(0)
            sd = src.reshape(-1, src.shape[-1]).std(0)
            w = (raw - mu) / np.where(sd > 0, sd, 1.0)
        else:
            raise KeyError(mode)
        out[c] = {**d, "windows": w.astype(np.float32)}
    return out


SCALINGS = ["minmax_ref", "none", "zscore"]

_WIN_CACHE: dict = {}


def build_windows(strategy: str, transform: str, prep: dict, clients=None, cap=None):
    """(fl_windows, scalers, fl_cfg) for one (placement, transform, prep params) cell."""
    key = (CURRENT_WORLD, strategy, transform, tuple(sorted(prep.items())),
          tuple(clients or ()), cap)
    if key in _WIN_CACHE:
        return _WIN_CACHE[key]
    fl = copy.deepcopy(FL_BASE)
    fl["preprocessing"].update(prep)
    frames = CLIENTS[strategy]
    if clients:
        frames = {c: frames[c] for c in clients}
    frames = apply_transform(frames, transform, STEPS_DAY)
    fw, sc, _ = preprocess_clients(frames, fl, TIME)
    if cap:
        for c, d in fw.items():
            n = len(d["windows"])
            if n > cap:
                i = np.linspace(0, n - 1, cap).round().astype(int)
                d["windows"], d["labels"] = d["windows"][i], d["labels"][i]
                d["window_start_step"] = d["window_start_step"][i]
    _WIN_CACHE[key] = (fw, sc, fl)
    return _WIN_CACHE[key]


PREP_BASE = dict(interval_agg_h=FL_BASE["preprocessing"]["interval_agg_h"],
                 window_size=FL_BASE["preprocessing"]["window_size"],
                 step_size=FL_BASE["preprocessing"]["step_size"])

fw, sc, _fl = build_windows("manual", "none", PREP_BASE)
print("base cell:", {c: d["windows"].shape for c, d in fw.items()})
inv = rescale_windows(fw, sc, "none", _fl)
c0 = next(iter(fw))
print(f"inversion check ({c0}) -- physical range "
      f"{inv[c0]['windows'].min():.3f}..{inv[c0]['windows'].max():.3f}  "
      f"vs scaled {fw[c0]['windows'].min():.2f}..{fw[c0]['windows'].max():.2f}")

base cell: {'District_A': (9528, 84, 3), 'District_B': (9528, 84, 3), 'District_C': (9528, 84, 3), 'District_D': (9528, 84, 3), 'District_E': (9528, 84, 3)}
inversion check (District_A) -- physical range 0.527..13.689  vs scaled -1.07..1.15


## 4. What am I feeding the model?

Pick 2-3 clients, a channel, the preprocessing parameters, and scrub the window index. Rows:

1. **client series** after the transform, with the selected window shaded -- the full context.
2. **the window the model sees**, one line per client, in whatever units the scaling leaves.

Windows are rebuilt on change for the selected clients only, and cached.

In [9]:
import ipywidgets as W
import matplotlib.pyplot as plt
from IPython.display import display



print(f"{WORLD['tag']}  |  drift {TRUTH_CLIENT} from month {DRIFT_START}  |  "
      f"{TIME['n_months']} months @ {TIME['resolution_h']}h  |  steps/day {STEPS_DAY}")


ALL_CLIENTS = sorted(CLIENTS["manual"])
_c = W.SelectMultiple(options=ALL_CLIENTS, value=tuple(ALL_CLIENTS[:3]), rows=5,
                      description="clients")
_s = W.Dropdown(options=STRATEGIES, value="manual", description="placement")
_t = W.Dropdown(options=TRANSFORMS, value="none", description="transform")
_k = W.Dropdown(options=SCALINGS, value="minmax_ref", description="scaling")
_a = W.Dropdown(options=[1, 2, 3, 4, 6, 8], value=PREP_BASE["interval_agg_h"], description="agg h")
_w = W.Dropdown(options=[12, 24, 48, 84, 168], value=PREP_BASE["window_size"], description="window")
_p = W.Dropdown(options=[1, 3, 6, 12, 24], value=PREP_BASE["step_size"], description="stride")
_ch = W.Dropdown(options=["ch 0", "ch 1", "ch 2"], value="ch 0", description="channel")
_i = W.IntSlider(value=0, min=0, max=100, step=1, description="window", continuous_update=False,
                 layout=W.Layout(width="95%"))
_o = W.Output()


def _draw(*_):
    clients = list(_c.value)[:3]
    if not clients:
        return
    prep = dict(interval_agg_h=_a.value, window_size=_w.value, step_size=_p.value)
    try:
        fw, sc, fl = build_windows(_s.value, _t.value, prep, clients=tuple(clients))
        fw = rescale_windows(fw, sc, _k.value, fl)
    except Exception as err:
        with _o:
            _o.clear_output(wait=True); print(f"{type(err).__name__}: {err}")
        return

    j = int(_ch.value.split()[1])
    n = min(len(fw[c]["windows"]) for c in clients)
    _i.max = n - 1
    idx = min(_i.value, n - 1)
    src = apply_transform({c: CLIENTS[_s.value][c] for c in clients}, _t.value, STEPS_DAY)

    with _o:
        _o.clear_output(wait=True)
        fig, ax = plt.subplots(2, 1, figsize=(12, 6),
                               gridspec_kw={"height_ratios": [2, 1]})
        for c in clients:
            d, f = fw[c], src[c]
            cols = [x for x in f.columns if x not in ("timestamp", "month")]
            jj = min(j, len(cols) - 1)
            ax[0].plot(f[cols[jj]].to_numpy(), lw=.4, alpha=.75,
                       label=f"{c} · {cols[jj]}")
            start = int(d["window_start_step"][idx])
            span = _w.value * _a.value // int(TIME["resolution_h"])
            ax[0].axvspan(start, start + span, color="k", alpha=.10)
            ax[1].plot(d["windows"][idx, :, jj], lw=1.2, marker=".", ms=3, label=c)
        ax[0].legend(fontsize=7, ncol=3); ax[0].set_title(
            f"{_s.value} · {_t.value} · agg {_a.value}h · window {_w.value} · stride {_p.value}"
            f"  |  {n} windows/client")
        ax[1].set_title(f"window {idx} as the model sees it ({_k.value})"); ax[1].legend(fontsize=7)
        for a in ax:
            a.grid(alpha=.25)
        fig.tight_layout(); plt.show()

        rows = [{"client": c, "windows": len(fw[c]["windows"]),
                 "shape": str(fw[c]["windows"].shape[1:]),
                 "min": float(fw[c]["windows"].min()), "max": float(fw[c]["windows"].max()),
                 "std": float(fw[c]["windows"].std())} for c in clients]
        display(pd.DataFrame(rows).round(3))


for ctl in (_c, _s, _t, _k, _a, _w, _p, _ch, _i):
    ctl.observe(_draw, names="value")
display(W.VBox([W.HBox([_c, W.VBox([_s, _t, _k]), W.VBox([_a, _w, _p, _ch])]), _i, _o]))
_draw()


Dli__LI_LR_LR_LR_LI__baseline  |  drift District_D from month 6  |  30 months @ 1h  |  steps/day 24


## 5. What each transform keeps -- no training

`fl_preprocessing` does hourly -> `interval_agg_h` aggregation -> sliding window -> per-client
MinMax on the commissioning months. Nothing else. The transforms on offer, and what each risks:

| axis | does | risks |
|---|---|---|
| `slog` | signed `log1p` | flattens the amplitude that separates regimes |
| `diff` | first difference | removes the level -- which *is* the drift signal |
| `deseason_own` | per-channel `(month,hour,weekend)` mean removed | invariant 5; also deletes the client's monthly level |
| `deseason_shared` | top-4 PCs of the **pooled** cell-profile matrix removed | keeps each channel's level; assumes the confound is low-rank |
| `scaling=none/zscore` | invert the per-client MinMax | clients end on different scales; FedAvg must absorb it |

Rather than rank these by a trained score, decompose the **raw** client series into three
orthogonal components and ask which one carries the regime, per transform:

`level` (mean of the daily profile -- the consumption ladder), `amp` (its SD -- diurnal
amplitude, invariant 3's peak factor), `shape` (the amplitude-normalised curve).

One observation **per client** (aggregating channels first, or within-client channel variance
swamps the between-regime term), scored on the **commissioning months** only -- after the mover
drifts its `consumption_map` token is stale. `*_lift` is eta2 minus its own label-shuffle null;
read lift, never raw eta2, because the null sits near `1/(n-1)`.

In [11]:
INIT_MONTHS_N = 3                     # commissioning slice
CONTROLS = ("control_rot",)           # channel-space rotation; a data-side control


def _regimes_of(world_row) -> dict:
    cmap = world_row.get("consumption_map") or ""
    return {f"District_{chr(ord('A') + i)}": t
            for i, t in enumerate(cmap.split("_"))} if cmap else {}


REGIMES = _regimes_of(WORLD)


def apply_transform_ext(frames: dict, name: str, steps_day: int, seed: int = 0) -> dict:
    """apply_transform plus a random channel rotation as a data-side control."""
    if name not in CONTROLS:
        return apply_transform(frames, name, steps_day)
    rng, out = np.random.default_rng(seed), {}
    for c, f in frames.items():
        g = f.copy()
        cols = [x for x in g.columns if x not in ("timestamp", "month")]
        V = g[cols].to_numpy(float)
        Q, _ = np.linalg.qr(rng.normal(size=(V.shape[1], V.shape[1])))
        g[cols] = (V - V.mean(0)) @ Q + V.mean(0)
        out[c] = g
    return out


def _components(frames: dict, steps_day: int, months=None):
    """ONE observation per client -> (meta[level, amp], shapes matrix).

    Channels are z-scored over time then averaged, so channels on different physical
    scales (mca vs L/s) contribute comparably.
    """
    rows, shapes = [], []
    for c in sorted(frames):
        f = frames[c]
        cols = [x for x in f.columns if x not in ("timestamp", "month")]
        mon, hour = f["month"].to_numpy(), np.arange(len(f)) % steps_day
        keep_m = np.isin(mon, list(months)) if months is not None else np.ones(len(f), bool)
        V = f[cols].to_numpy(float)
        V = (V - V.mean(0)) / np.where(V.std(0) > 0, V.std(0), 1.0)
        v = V.mean(1)
        sel = keep_m & np.isfinite(v)
        if sel.sum() < steps_day:
            continue
        cnt = np.maximum(np.bincount(hour[sel], minlength=steps_day), 1)
        p = np.bincount(hour[sel], v[sel], steps_day) / cnt
        lev, amp = float(p.mean()), float(p.std())
        rows.append(dict(client=c, level=lev, amp=amp))
        shapes.append((p - lev) / (amp if amp > 0 else 1.0))
    return pd.DataFrame(rows), np.asarray(shapes)


def _eta2(X, g) -> float:
    X = np.asarray(X, float)
    X = X.reshape(-1, 1) if X.ndim == 1 else X
    ok = np.isfinite(X).all(1)
    X, g = X[ok], np.asarray(g)[ok]
    if len(X) < 3 or len(np.unique(g)) < 2:
        return np.nan
    tot = float(((X - X.mean(0)) ** 2).sum())
    if tot <= 0:
        return np.nan
    btw = sum(len(X[g == u]) * float(((X[g == u].mean(0) - X.mean(0)) ** 2).sum())
              for u in np.unique(g))
    return btw / tot


def _shuffle_null(fn, clients, regimes: dict, n_max: int = 2000, seed: int = 0):
    """(real, p, null_mean, n_assign) for any statistic of a district->token map.

    The null holds the DATA fixed and permutes the label correspondence -- the only
    honest floor. Exhaustive while the assignment space is small.
    """
    ks = sorted({c for c in clients if c in regimes})
    if len(ks) < 3:
        return np.nan, np.nan, np.nan, 0
    real = fn({c: regimes[c] for c in ks})
    toks = [regimes[c] for c in ks]
    perms, seen = [], set()
    for p in itertools.permutations(toks):
        if p not in seen:
            seen.add(p); perms.append(p)
        if len(perms) >= n_max:
            break
    null = [fn(dict(zip(ks, p))) for p in perms]
    null = np.array([x for x in null if np.isfinite(x)])
    p_val = float((null >= real).mean()) if len(null) and np.isfinite(real) else np.nan
    return real, p_val, (float(null.mean()) if len(null) else np.nan), len(seen)


def prep_diagnostics(strategy="manual", transforms=None, controls=True,
                     init_n=INIT_MONTHS_N) -> pd.DataFrame:
    names = list(transforms or TRANSFORMS) + (list(CONTROLS) if controls else [])
    any_c = next(iter(CLIENTS[strategy]))
    init = sorted(int(m) for m in CLIENTS[strategy][any_c]["month"].unique())[:init_n]
    rows = []
    for t in names:
        fr = apply_transform_ext(CLIENTS[strategy], t, STEPS_DAY)
        sig = {}
        for c, f in fr.items():
            cols = [x for x in f.columns if x not in ("timestamp", "month")]
            V = f[cols].to_numpy(float)
            sig[c] = ((V - V.mean(0)) / np.where(V.std(0) > 0, V.std(0), 1.0)).mean(1)
        ks = sorted(sig)
        R = np.abs(np.corrcoef(np.column_stack([sig[k] for k in ks]).T))
        rec = {"transform": t,
               "cross_client_r": float(np.nanmean(R[np.triu_indices(len(ks), 1)]))}

        meta, shapes = _components(fr, STEPS_DAY, months=init)
        if len(meta):
            cl = meta["client"].tolist()
            for tag, X in (("level", meta["level"].to_numpy()),
                           ("amp", meta["amp"].to_numpy()), ("shape", shapes)):
                e, p, nm, n = _shuffle_null(
                    lambda mp, X=X: _eta2(X, np.array([mp.get(c, "?") for c in cl])), cl, REGIMES)
                rec[f"{tag}_lift"] = e - nm if np.isfinite(e) and np.isfinite(nm) else np.nan
                rec[f"{tag}_p"], rec["n_assign"] = p, n
        rows.append(rec)
    return pd.DataFrame(rows)


def centralized_ceiling(strategy="manual", init_n=INIT_MONTHS_N) -> pd.DataFrame:
    """The district-mean daily profile pooled centrally -- nothing federated should beat it."""
    frames = {}
    for c, f in CLIENTS[strategy].items():
        cols = [x for x in f.columns if x not in ("timestamp", "month")]
        V = f[cols].to_numpy(float)
        Z = (V - V.mean(0)) / np.where(V.std(0) > 0, V.std(0), 1.0)
        frames[c] = f[["timestamp", "month"]].assign(district_mean=Z.mean(1))
    init = sorted(int(m) for m in next(iter(frames.values()))["month"].unique())[:init_n]
    meta, shapes = _components(frames, STEPS_DAY, months=init)
    cl, out = meta["client"].tolist(), {"source": "centralized daily profile"}
    for tag, X in (("level", meta["level"].to_numpy()), ("amp", meta["amp"].to_numpy()),
                   ("shape", shapes)):
        e, p, nm, _ = _shuffle_null(
            lambda mp, X=X: _eta2(X, np.array([mp.get(c, "?") for c in cl])), cl, REGIMES)
        out[f"{tag}_lift"], out[f"{tag}_p"] = e - nm, p
    return pd.DataFrame([out])


DIAG_PREP = prep_diagnostics()
display(pd.concat([DIAG_PREP, centralized_ceiling()], ignore_index=True)
        [["transform", "source", "cross_client_r", "level_lift", "level_p",
          "amp_lift", "amp_p", "shape_lift", "shape_p"]].round(3))
n_a = int(DIAG_PREP["n_assign"].max())
print(f"regimes {REGIMES} | months {init_n if (init_n := INIT_MONTHS_N) else 0}..  | "
      f"{n_a} distinct assignments -> floor p = {1 / max(1, n_a):.2f}")
print("cross_client_r LOW is better (less shared exogenous drive).")
print("*_lift > 0 means the component carries regime beyond the shuffle null; "
      "control_rot is the data-side floor.")
print("NOTE amp_lift is contaminated -- a channel rotation inflates it, so do not rank on it.")

,transform,source,cross_client_r,level_lift,level_p,amp_lift,amp_p,shape_lift,shape_p
0,none,NaN,0.599,-0.209,0.8,0.721,0.1,0.226,0.2
1,slog,NaN,0.611,-0.223,0.9,0.660,0.1,0.156,0.2
2,diff,NaN,0.493,0.467,0.2,0.742,0.1,0.353,0.1
3,deseason_own,NaN,0.144,0.073,0.4,-0.118,0.5,0.062,0.3
4,deseason_shared,NaN,0.102,-0.181,0.7,0.041,0.4,-0.114,0.9
5,control_rot,NaN,0.712,-0.213,0.8,-0.248,0.9,-0.244,1.0
6,NaN,centralized daily profile,NaN,-0.209,0.9,-0.200,0.8,0.226,0.2


regimes {'District_A': 'LI', 'District_B': 'LR', 'District_C': 'LR', 'District_D': 'LR', 'District_E': 'LI'} | months 3..  | 10 distinct assignments -> floor p = 0.10
cross_client_r LOW is better (less shared exogenous drive).
*_lift > 0 means the component carries regime beyond the shuffle null; control_rot is the data-side floor.
NOTE amp_lift is contaminated -- a channel rotation inflates it, so do not rank on it.


## 6. Scoring a cell

Two families of score, never mixed:

* **latent** -- `sil_district` / `sil_month` (cosine silhouette), `month_within_district`
  (month decoded *inside* each client, so it cannot ride the client fingerprint), `eff_dim`.
* **drift** -- `compute_drift_signals` on the prototypes, verbatim: `drift_rank` of the true
  drifted district (1 is correct), `drift_sep` in robust units of the other clients' spread
  (median/MAD, so one collapsed-variance client can't blow the ratio up), `drift_gap` (target
  mean delta / next-largest, scale-free) and `others_sd` so a near-zero denominator is visible
  rather than silently inflating `drift_sep`.

In [12]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def _probe(Z, y, seed=0):
    u, cnt = np.unique(y, return_counts=True)
    if len(u) < 2:
        return np.nan
    Ztr, Zte, ytr, yte = train_test_split(Z, y, test_size=.3, random_state=seed,
                                          stratify=y if cnt.min() >= 2 else None)
    return float(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500))
                 .fit(Ztr, ytr).score(Zte, yte))


def latent_metrics(lt, seed=0, sil_n=2000, probe_n=6000) -> dict:
    f = [c for c in lt.columns if c.startswith("f") and c[1:].isdigit()]
    Z, d, m = lt[f].to_numpy(float), lt["district"].to_numpy(), lt["month"].to_numpy()
    rng = np.random.default_rng(seed)
    i = rng.choice(len(Z), min(sil_n, len(Z)), replace=False)
    p = rng.choice(len(Z), min(probe_n, len(Z)), replace=False)
    ev = PCA().fit(Z).explained_variance_ratio_
    within = [a for a in (_probe(Z[d == c], m[d == c], seed) for c in np.unique(d))
              if not np.isnan(a)]
    return dict(sil_district=float(silhouette_score(Z[i], d[i], metric="cosine")),
                sil_month=float(silhouette_score(Z[i], m[i], metric="cosine")),
                month_within_district=float(np.mean(within)) if within else np.nan,
                probe_district=_probe(Z[p], d[p], seed),
                eff_dim=float(1.0 / (ev ** 2).sum()))


def _auroc(y, s):
    from scipy import stats as sps
    y, s = np.asarray(y, bool), np.asarray(s, float)
    if y.all() or not y.any():
        return np.nan
    r = sps.rankdata(s); n1, n0 = y.sum(), (~y).sum()
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def drift_metrics(ph: pd.DataFrame, fl: dict, truth=None, start=None) -> dict:
    truth, start = truth or TRUTH_CLIENT, DRIFT_START if start is None else start
    ds = compute_drift_signals(ph, fl)
    if truth not in set(ds["client"]):
        return dict(drift_rank=np.nan, drift_sep=np.nan, drift_gap=np.nan,
                   others_sd=np.nan, drift_auroc=np.nan)
    post = ds[ds["month"] >= start]
    m = post.groupby("client")["delta_first"].mean().abs()
    others = m.drop(truth)
    mad = float(np.median(np.abs(others - others.median()))) or float(others.mean()) or 1.0
    tgt = ds[ds["client"] == truth]
    return dict(
        drift_rank=float(m.rank(ascending=False)[truth]),
        drift_sep=float((m[truth] - others.median()) / mad),
        drift_gap=float(m[truth] / (others.max() or np.nan)),
        others_sd=float(others.std(ddof=0)),
        drift_auroc=_auroc(tgt["month"].to_numpy() >= start, tgt["delta_first"].abs().to_numpy()))

## 7. Persistence -- `latent_sandbox/`

`cell_id` is deterministic from the cell spec, so re-running a grid is a cache hit.

```
latent_sandbox/
  runs.csv                     # registry: one row per (world_id, cell_id)
  clustering_by_phase.csv      # SS.13
  <world_id>/<cell_id>/
    config.json  metrics.json
    latent_trajectories        prototype_history (carries `round`)   drift_signals
    latents_by_round           metrics_by_round        (SS.7a, optional)
    similarity_by_phase        clusters_by_phase       (SS.13)
```

`run_cell` is the only entry point -- SS.8, SS.11 and SS.12 all go through it, so a cell trains
once however many places ask for it.

In [13]:
def cell_id_for(strategy, transform, scaling, prep) -> str:
    return (f"{strategy}__{transform}__{scaling}"
            f"__a{prep['interval_agg_h']}w{prep['window_size']}s{prep['step_size']}")


def _write_table(df: pd.DataFrame, path_base: pathlib.Path) -> None:
    try:
        df.to_parquet(path_base.with_suffix(".parquet"), index=False)
    except Exception:
        df.to_csv(path_base.with_suffix(".csv.gz"), index=False)


def _read_table(path_base: pathlib.Path):
    for suf in (".parquet", ".csv.gz"):
        p = path_base.with_suffix(suf)
        if p.exists():
            return pd.read_parquet(p) if suf == ".parquet" else pd.read_csv(p)
    return None


CELL_TABLES = ("latent_trajectories", "prototype_history", "drift_signals",
               "latents_by_round", "metrics_by_round",
               "similarity_by_phase", "clusters_by_phase")


def save_cell(world_id, cell_id, spec, lt, ph, drift, metrics, **tables) -> pathlib.Path:
    out = SANDBOX / world_id / cell_id
    out.mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(spec, indent=2))
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    for name, df in (("latent_trajectories", lt), ("prototype_history", ph),
                     ("drift_signals", drift)):
        _write_table(df, out / name)
    for name, df in tables.items():
        if df is not None:
            key = "latents_by_round" if name == "snaps" else name
            _write_table(df.reset_index() if df.index.name else df, out / key)

    reg_path = SANDBOX / "runs.csv"
    reg = pd.read_csv(reg_path) if reg_path.exists() else pd.DataFrame()
    if len(reg):
        reg = reg[~((reg.world_id == world_id) & (reg.cell_id == cell_id))]
    row = {"world_id": world_id, "cell_id": cell_id,
          "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), **metrics}
    pd.concat([reg, pd.DataFrame([row])], ignore_index=True).to_csv(reg_path, index=False)
    return out


def load_cell(world_id, cell_id) -> dict | None:
    d = SANDBOX / world_id / cell_id
    if not (d / "metrics.json").exists():
        return None
    out = dict(spec=json.loads((d / "config.json").read_text()),
              metrics=json.loads((d / "metrics.json").read_text()))
    for name in CELL_TABLES:
        out[name] = _read_table(d / name)
    return out


def save_analysis(world_id, cell_id, **tables) -> pathlib.Path:
    out = SANDBOX / world_id / cell_id
    if not out.exists():
        raise FileNotFoundError(f"{out} does not exist -- run SS.8 first")
    for name, df in tables.items():
        if df is not None:
            _write_table(df.reset_index() if df.index.name else df, out / name)
    return out


def load_registry() -> pd.DataFrame:
    p = SANDBOX / "runs.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

### 7a. The round axis

Two different per-round objects, and they are not interchangeable:

* **`prototype_history`** already carries `round` -- `FPLTrainer` emits it. Every per-round
  *prototype* analysis (SS.14 similarity, SS.15 migration) reads this and needs **no retraining**,
  so cells trained before this section still work.
* **`latents_by_round`** is new and needs `SnapshotFPLTrainer`: the same fixed subset of windows
  encoded after every round, so the latent *cloud* can be followed frame to frame. `round = -1`
  is the shared untrained init -- the only true control in the notebook.

Snapshots are taken post-local-update, **pre-FedAvg** (the personalized view), exactly as in
`aer_federated_sandbox_plot`. `FPLTrainer`'s protocol, seeding and FedAvg are untouched.

In [14]:
class SnapshotFPLTrainer(FPLTrainer):
    """FPLTrainer + fixed-subset latent snapshots per round. Protocol untouched."""

    def __init__(self, client_windows, fl, seed, every=1, points=900):
        super().__init__(client_windows, fl, seed)
        self.starts = {c: client_windows[c]["window_start_step"] for c in self.clients}
        self.snap_every = int(every or 0)                 # None -> 0 (snapshots off)
        points = 900 if points is None else int(points)   # None -> the default, not a crash
        self.n_rounds, self.snap_rows = 0, []

        rng = np.random.default_rng(seed)
        per = max(1, points // len(self.clients))
        self.snap_idx = {
            c: np.sort(rng.choice(len(self.data[c][0]),
                                  min(per, len(self.data[c][0])), replace=False))
            for c in self.clients
        }
        if self.snap_every:
            for c in self.clients:
                self._snapshot(c, -1)                      # shared untrained init

    @torch.no_grad()
    def _snapshot(self, client, round_idx):
        model, bs = self.models[client], self.cfg_t["batch_size"]
        mode = model.training
        model.eval()
        w = self.data[client][0][self.snap_idx[client]]
        z = torch.cat([model.encode(w[i:i + bs, 1:-1])
                       for i in range(0, len(w), bs)]).cpu().numpy()
        model.train(mode)

        idx = self.snap_idx[client]
        df = pd.DataFrame(z, columns=[f"f{i}" for i in range(z.shape[1])])
        df.insert(0, "month", self.data[client][1].cpu().numpy()[idx])
        df.insert(0, "window", self.starts[client][idx])
        df.insert(0, "round", round_idx)
        df.insert(0, "district", client)
        self.snap_rows.append(df)

    # ------------------------------------------------------------- hooks
    def _local_update(self, client, round_idx):
        protos = super()._local_update(client, round_idx)
        if self.snap_every and (round_idx % self.snap_every == 0
                                or round_idx == self.n_rounds - 1):
            self._snapshot(client, round_idx)
        return protos

    def train(self, rounds):
        self.n_rounds = rounds
        return super().train(rounds)

    def snapshots(self):
        return pd.concat(self.snap_rows, ignore_index=True) if self.snap_rows else None


SNAPSHOT_EVERY  = 1 if globals().get("SNAPSHOT_EVERY") is None else SNAPSHOT_EVERY
SNAPSHOT_POINTS = 900 if globals().get("SNAPSHOT_POINTS") is None else SNAPSHOT_POINTS
print("snapshots:", SNAPSHOT_EVERY, SNAPSHOT_POINTS)
ROUND_METRICS   = True  # True = latent_metrics per round (adds ~1-2 s per round per cell)

snapshots: 1 900


In [15]:
def set_world_context(world_row):
    """Point ART/CLIENTS/etc at a different world. Called by run_cell when needed."""
    global ART, TIME, STEPS_DAY, FL_BASE, TRUTH_CLIENT, DRIFT_START
    global CLIENTS, PLACEMENTS, _WIN_CACHE, CURRENT_WORLD
    art = P.load_world(world_row["dir"])
    t = world_time(art)                                  # numeric, whatever the manifest says
    art["params"]["time"] = t                            # so preprocess_clients sees it too
    ART, TIME, STEPS_DAY = art, t, int(_num(art.get("steps_day"), 24.0 / t["resolution_h"]))
    FL_BASE = world_fl(art)
    TRUTH_CLIENT = world_row.get("drift_district")
    DRIFT_START = drift_start(world_row["dir"], t["n_months"])

    cands = P.candidate_table(ART["wn"], ART["districts"])
    feats, dist = P.topology_features(ART["wn"], ART["districts"], cands)
    stats = P.residual_stats(ART["pressures"], ART["flows"], cands, STEPS_DAY)
    PLACEMENTS = {k: v for k, v in P.strategies(cands, feats, stats, dist, ART["params"],
                                                seed=SEED).items() if k in STRATEGIES}
    missing = [s for s in STRATEGIES if s not in PLACEMENTS]
    if missing:
        raise RuntimeError(f"placements {missing} not offered by P.strategies "
                           f"(have {sorted(PLACEMENTS)})")
    CLIENTS = {s: clients_from_placement(ART, pl, kinds=("flow",))
               for s, pl in PLACEMENTS.items()}
    for s, frames in CLIENTS.items():
        assert_equal_channels(frames)
    _WIN_CACHE = {}
    CURRENT_WORLD = world_row["sim_hash"]


def train_cell(strategy, transform, scaling, prep, rounds=None, cap=None,
              snapshot_every=None, snapshot_points=None) -> dict:
    """Pure compute: windows -> SnapshotFPLTrainer -> latents, prototypes, snapshots."""
    fw, sc, fl = build_windows(strategy, transform, prep, cap=cap)
    fw = rescale_windows(fw, sc, scaling, fl)
    fl = copy.deepcopy(fl)
    fl["training"]["rounds"] = int(_num(rounds, _num(fl["training"].get("rounds"), 15)))
    every = SNAPSHOT_EVERY if snapshot_every is None else snapshot_every
    pts = snapshot_points if snapshot_points is not None else (SNAPSHOT_POINTS or 900)

    tr = SnapshotFPLTrainer(fw, fl, effective_fl_seed(fl, SEED), every=every, points=pts)
    t0 = time.time()
    tr.train(fl["training"]["rounds"])

    res_h = float(TIME["resolution_h"])

    def _tag(df):
        df = df.copy()
        df["hour"] = (df["window"] * res_h % 24).astype(int)
        df["drift_status"] = np.where(
            (df["district"] == TRUTH_CLIENT) & (df["month"] >= DRIFT_START), "drifted", "other")
        return df

    lt = _tag(tr.latent_trajectories(fw))
    snaps = tr.snapshots()
    snaps = None if snaps is None else _tag(snaps)
    ph = pd.DataFrame(tr.local_proto_rows)
    return dict(lt=lt, ph=ph, snaps=snaps, drift=compute_drift_signals(ph, fl), fl=fl, fw=fw,
               seconds=round(time.time() - t0, 1))


def score_cell(out: dict, strategy, transform, scaling, prep) -> dict:
    return dict(strategy=strategy, transform=transform, scaling=scaling, **prep,
               n_windows=int(sum(len(d["windows"]) for d in out["fw"].values())),
               n_rounds=int(_num(out["fl"]["training"].get("rounds"), 0)),
               **latent_metrics(out["lt"]), **drift_metrics(out["ph"], out["fl"]),
               seconds=out["seconds"])


def metrics_by_round(snaps: pd.DataFrame | None) -> pd.DataFrame | None:
    """latent_metrics per snapshot round. Round -1 is the untrained control."""
    if snaps is None:
        return None
    return pd.DataFrame([{"round": int(r), "n": len(g), **latent_metrics(g)}
                         for r, g in snaps.groupby("round")])


def run_cell(world_row, strategy, transform, scaling, prep, rounds=None, cap=None,
             overwrite=False, save=True, return_tables=False, verbose=False) -> dict:
    """The single entry point. Trains + scores + persists a cell, or loads it from disk."""
    if world_row["sim_hash"] != CURRENT_WORLD:
        set_world_context(world_row)
    wid, cid = world_row["sim_hash"], cell_id_for(strategy, transform, scaling, prep)

    cached = None if overwrite else load_cell(wid, cid)
    if cached is not None:
        row = {**cached["metrics"], "world_id": wid, "cell_id": cid, "tag": world_row["tag"],
              "cache_hit": True}
        if return_tables:
            row["_tables"] = cached
        if verbose:
            print(f"  [cached] {cid}")
        return row

    out = train_cell(strategy, transform, scaling, prep, rounds=rounds, cap=cap)
    row = score_cell(out, strategy, transform, scaling, prep)
    mbr = metrics_by_round(out["snaps"]) if ROUND_METRICS else None
    if save:
        save_cell(wid, cid, dict(strategy=strategy, transform=transform, scaling=scaling,
                                 prep=prep, rounds=rounds, cap=cap,
                                 snapshot_every=SNAPSHOT_EVERY),
                 out["lt"], out["ph"], out["drift"], row,
                 snaps=out["snaps"], metrics_by_round=mbr)
    row = {**row, "world_id": wid, "cell_id": cid, "tag": world_row["tag"], "cache_hit": False}
    if return_tables:
        row["_tables"] = {"latent_trajectories": out["lt"], "prototype_history": out["ph"],
                          "drift_signals": out["drift"], "latents_by_round": out["snaps"],
                          "metrics_by_round": mbr}
    if verbose:
        print("  ".join(f"{k}={v}" for k, v in row.items()
                        if k not in prep and not k.startswith("_")))
    return row

## 8. The grid, one world

Two blocks, ~36 cells. Block A holds the window geometry fixed and sweeps
`placement x transform x scaling`; block B holds the transform fixed and sweeps the geometry.
Crossing both fully would be 90 trains for little extra information -- the interaction that
matters is placement x transform.

`CAP` subsamples windows per client. Set it to `None` for the honest run; keep it while you are
still deciding the grid. Re-running this cell after the first pass is a disk-cache read, not a
retrain -- pass `overwrite=True` to `run_cell` (or delete the cell's directory) to force one.

In [12]:
CAP    = 3000        # windows per client; None = all
ROUNDS = None        # None = the world's own fl.training.rounds

PREP_BASE = {'interval_agg_h': 3, 'window_size': 84, 'step_size': 12}

PREP_GEOM = [PREP_BASE,
             dict(interval_agg_h=2, window_size=24, step_size=2),
             dict(interval_agg_h=2, window_size=24, step_size=6),
             dict(interval_agg_h=3, window_size=84, step_size=24),
             dict(interval_agg_h=4, window_size=48, step_size=12)]

GRID = ([dict(strategy=s, transform=t, scaling=k, prep=PREP_BASE)
         for s, t, k in itertools.product(STRATEGIES, TRANSFORMS, ["minmax_ref", "none"])]
        + [dict(strategy=s, transform="none", scaling="minmax_ref", prep=g)
           for s, g in itertools.product(STRATEGIES, PREP_GEOM[1:])])

print(f"{len(GRID)} cells")
pd.DataFrame([{**{k: v for k, v in g.items() if k != 'prep'}, **g['prep']} for g in GRID]).head(8)

42 cells


,strategy,transform,scaling,interval_agg_h,window_size,step_size
0,manual,none,minmax_ref,3,84,12
1,manual,none,none,3,84,12
2,manual,slog,minmax_ref,3,84,12
3,manual,slog,none,3,84,12
4,manual,diff,minmax_ref,3,84,12
5,manual,diff,none,3,84,12
6,manual,deseason_own,minmax_ref,3,84,12
7,manual,deseason_own,none,3,84,12


In [15]:
rows, fails = [], []
for n, g in enumerate(GRID, 1):
    tag = f"{g['strategy'][:4]}/{g['transform']}/{g['scaling']}/a{g['prep']['interval_agg_h']}w{g['prep']['window_size']}s{g['prep']['step_size']}"
    try:
        r = run_cell(WORLD, **g, rounds=ROUNDS, cap=CAP)
        rows.append(r)
        flag = "cached" if r["cache_hit"] else f"{r['seconds']}s"
        print(f"[{n}/{len(GRID)}] {tag:<44} rank {r['drift_rank']:.0f}  "
              f"sep {r['drift_sep']:+.2f}  mwd {r['month_within_district']:.3f}  {flag}")
    except Exception as exc:
        fails.append({"cell": tag, "error": f"{type(exc).__name__}: {exc}"})
        print(f"[{n}/{len(GRID)}] {tag:<44} FAILED {type(exc).__name__}: {exc}")

RESULTS = pd.DataFrame(rows)
if fails:
    display(pd.DataFrame(fails))
print(f"saved under {SANDBOX}/{WORLD['sim_hash']}/  |  registry: {SANDBOX}/runs.csv")
RESULTS.sort_values(["drift_rank", "drift_sep"], ascending=[True, False]).head(12).round(3)

[1/42] manu/none/minmax_ref/a3w84s12                rank 2  sep +1.36  mwd 0.160  13.4s
[2/42] manu/none/none/a3w84s12                      rank 1  sep +41.15  mwd 0.203  43.0s
[3/42] manu/slog/minmax_ref/a3w84s12                rank 1  sep +3.44  mwd 0.176  47.0s
[4/42] manu/slog/none/a3w84s12                      rank 4  sep -1.00  mwd 0.172  48.5s
[5/42] manu/diff/minmax_ref/a3w84s12                rank 1  sep +2.39  mwd 0.089  44.7s
[6/42] manu/diff/none/a3w84s12                      rank 2  sep +23.58  mwd 0.068  47.0s
[7/42] manu/deseason_own/minmax_ref/a3w84s12        rank 3  sep -0.14  mwd 0.028  50.3s
[8/42] manu/deseason_own/none/a3w84s12              rank 1  sep +2.95  mwd 0.028  41.6s
[9/42] manu/deseason_shared/minmax_ref/a3w84s12     rank 3  sep -0.12  mwd 0.222  45.7s
[10/42] manu/deseason_shared/none/a3w84s12           rank 5  sep -2.32  mwd 0.260  45.6s
[11/42] vari/none/minmax_ref/a3w84s12                rank 1  sep +2.30  mwd 0.093  47.7s
[12/42] vari/none/none/a3w84

,strategy,transform,scaling,interval_agg_h,window_size,step_size,n_windows,n_rounds,sil_district,sil_month,...,drift_rank,drift_sep,drift_gap,others_sd,drift_auroc,seconds,world_id,cell_id,tag,cache_hit
1,manual,none,none,3,84,12,2390,15,0.968,-0.089,...,1.0,41.148,4.774,0.004,0.993,43.0,e0f8c48c1006,manual__none__none__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False
19,variance,deseason_shared,none,3,84,12,2390,15,0.999,-0.074,...,1.0,14.468,1.253,0.000,0.837,48.2,e0f8c48c1006,variance__deseason_shared__none__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False
18,variance,deseason_shared,minmax_ref,3,84,12,2390,15,-0.005,-0.203,...,1.0,10.128,2.082,0.004,0.847,46.0,e0f8c48c1006,variance__deseason_shared__minmax_ref__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False
32,manual,none,minmax_ref,3,84,24,1195,15,0.220,-0.194,...,1.0,5.693,1.041,0.011,0.938,22.4,e0f8c48c1006,manual__none__minmax_ref__a3w84s24,Dli__LI_LR_LR_LR_LI__baseline,False
33,manual,none,minmax_ref,4,48,12,2090,15,0.208,-0.261,...,1.0,3.904,1.036,0.009,0.972,26.8,e0f8c48c1006,manual__none__minmax_ref__a4w48s12,Dli__LI_LR_LR_LR_LI__baseline,False
30,manual,none,minmax_ref,2,24,2,15000,15,0.206,-0.202,...,1.0,3.739,1.411,0.044,1.000,108.1,e0f8c48c1006,manual__none__minmax_ref__a2w24s2,Dli__LI_LR_LR_LR_LI__baseline,False
2,manual,slog,minmax_ref,3,84,12,2390,15,0.316,-0.283,...,1.0,3.440,1.615,0.013,0.868,47.0,e0f8c48c1006,manual__slog__minmax_ref__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False
36,variance,none,minmax_ref,3,84,24,1195,15,0.133,-0.211,...,1.0,3.066,1.204,0.012,0.792,24.0,e0f8c48c1006,variance__none__minmax_ref__a3w84s24,Dli__LI_LR_LR_LR_LI__baseline,False
7,manual,deseason_own,none,3,84,12,2390,15,-0.297,-0.321,...,1.0,2.954,1.194,0.008,0.604,41.6,e0f8c48c1006,manual__deseason_own__none__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False
4,manual,diff,minmax_ref,3,84,12,2390,15,-0.105,-0.123,...,1.0,2.389,1.184,0.002,1.000,44.7,e0f8c48c1006,manual__diff__minmax_ref__a3w84s12,Dli__LI_LR_LR_LR_LI__baseline,False


## 9. Resume from disk

If you are picking this notebook back up in a fresh kernel, `RESULTS` rebuilds from the registry
with no retraining -- SS.7-8 only need re-running to add cells that were never saved.

In [1]:

list_cells(WORLD["1cc4d212d7b6"])

NameError: name 'list_cells' is not defined

In [17]:
def list_cells(world_id: str | None = None) -> pd.DataFrame:
    reg = load_registry()
    return reg if not world_id or not len(reg) else reg[reg.world_id == world_id]

RESULTS = list_cells(WORLD["sim_hash"])
print(f"{len(RESULTS)} cells on disk for {WORLD['tag']}")
RESULTS.sort_values(["drift_rank", "drift_sep"], ascending=[True, False])

42 cells on disk for Dli__LI_LR_LR_LR_LI__baseline


,world_id,cell_id,saved_at,strategy,transform,scaling,interval_agg_h,window_size,step_size,n_windows,...,sil_month,month_within_district,probe_district,eff_dim,drift_rank,drift_sep,drift_gap,others_sd,drift_auroc,seconds
1,e0f8c48c1006,manual__none__none__a3w84s12,2026-09-07 22:08:06,manual,none,none,3,84,12,2390,...,-0.089025,0.202778,1.000000,2.445560,1.0,41.148116,4.773547,4.151060e-03,0.993056,43.0
19,e0f8c48c1006,variance__deseason_shared__none__a3w84s12,2026-09-07 22:24:06,variance,deseason_shared,none,3,84,12,2390,...,-0.074031,0.188889,1.000000,2.697741,1.0,14.467799,1.253159,1.055797e-05,0.836806,48.2
18,e0f8c48c1006,variance__deseason_shared__minmax_ref__a3w84s12,2026-09-07 22:23:13,variance,deseason_shared,minmax_ref,3,84,12,2390,...,-0.202562,0.133333,0.868898,3.770886,1.0,10.128165,2.082185,3.922891e-03,0.847222,46.0
32,e0f8c48c1006,manual__none__minmax_ref__a3w84s24,2026-09-07 22:36:26,manual,none,minmax_ref,3,84,24,1195,...,-0.194245,0.163889,1.000000,3.845788,1.0,5.693159,1.040529,1.066643e-02,0.937500,22.4
33,e0f8c48c1006,manual__none__minmax_ref__a4w48s12,2026-09-07 22:37:01,manual,none,minmax_ref,4,48,12,2090,...,-0.261421,0.319048,1.000000,1.841964,1.0,3.903764,1.036080,8.574837e-03,0.972222,26.8
30,e0f8c48c1006,manual__none__minmax_ref__a2w24s2,2026-09-07 22:34:48,manual,none,minmax_ref,2,24,2,15000,...,-0.202197,0.347333,0.967778,5.167094,1.0,3.739343,1.410697,4.378593e-02,1.000000,108.1
2,e0f8c48c1006,manual__slog__minmax_ref__a3w84s12,2026-09-07 22:09:00,manual,slog,minmax_ref,3,84,12,2390,...,-0.282759,0.176389,0.994421,1.824054,1.0,3.440369,1.614738,1.281891e-02,0.868056,47.0
36,e0f8c48c1006,variance__none__minmax_ref__a3w84s24,2026-09-07 22:40:35,variance,none,minmax_ref,3,84,24,1195,...,-0.210779,0.127778,1.000000,3.745927,1.0,3.065526,1.204036,1.161834e-02,0.791667,24.0
7,e0f8c48c1006,manual__deseason_own__none__a3w84s12,2026-09-07 22:13:22,manual,deseason_own,none,3,84,12,2390,...,-0.320843,0.027778,0.789400,5.588991,1.0,2.953695,1.193717,7.878092e-03,0.604167,41.6
4,e0f8c48c1006,manual__diff__minmax_ref__a3w84s12,2026-09-07 22:10:47,manual,diff,minmax_ref,3,84,12,2390,...,-0.122733,0.088889,1.000000,2.270646,1.0,2.388999,1.183779,1.741548e-03,1.000000,44.7
